# 📗 판다스 기초 — 데이터프레임 조회·선택·정렬

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

> 📚 **pandas 공식 문서**: https://pandas.pydata.org/docs/ · 입문 가이드 [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) · API 검색 [API reference](https://pandas.pydata.org/docs/reference/index.html)

## 오늘의 목표
- [ ] **판다스(pandas)** 가 무엇이고 왜 데이터 분석의 기본 도구인지 이해한다.
- [ ] **데이터프레임(DataFrame)** 을 만들고 `head`·`info`·`describe` 로 훑어본다.
- [ ] 원하는 **열**과 **행**을 골라낸다 — `[]`, `loc`, `iloc`.
- [ ] **조건(불리언 필터)** 으로 원하는 행만 추린다.
- [ ] `sort_values` 로 데이터를 **정렬**한다.

## ⏪ 복습 — 지난 단원: 데이터를 수집했다, 이제 분석하자
지난 단원까지 우리는 **데이터를 모으는 법**을 배웠습니다.
- API 를 호출하고 JSON 을 파싱하고, CSV 파일을 열어 한 줄씩 읽었죠.
- 리스트·딕셔너리에 담고, 결측치를 `if` 로 걸러냈습니다.

그런데 수천·수만 줄짜리 표를 **순수 파이썬 반복문**으로 다루면 코드가 금세 길고 느려집니다.
표(table) 형태 데이터는 표 전용 도구로 다루는 게 정답이에요. 그게 바로 **판다스**입니다.
오늘은 카페 판매 기록을 가지고, 표를 **훑어보고·골라내고·정렬하는** 기본기를 익힙니다. 🚀

---
# 1. 데이터프레임 만들기와 훑어보기

## 왜 필요할까요?
**판다스**는 표(엑셀 시트 같은 2차원 데이터)를 다루는 파이썬 라이브러리입니다.
핵심 자료구조가 두 개예요.

> **일상 비유** — 엑셀을 떠올리세요.
> **시리즈(Series)** 는 엑셀의 **열 하나**(세로 한 줄), **데이터프레임(DataFrame)** 은 **시트 전체**(여러 열이 모인 표)입니다.

| 자료구조 | 무엇인가 | 비유 |
|---|---|---|
| `Series` | 이름표(인덱스)가 붙은 1차원 값들 | 엑셀 열 하나 |
| `DataFrame` | 여러 Series 가 모인 2차원 표 | 엑셀 시트 전체 |

표를 만드는 방법은 두 가지입니다.
- **딕셔너리 → DataFrame**: 파이썬 dict 로 직접 만들기 (작은 데이터)
- **`pd.read_csv`**: CSV 파일을 통째로 읽어 표로 만들기 (실무의 대부분)

표를 손에 넣으면 가장 먼저 **훑어봅니다**. 다음 도구들이 필수예요.

| 도구 | 하는 일 |
|---|---|
| `df.head(n)` / `df.tail(n)` | 위·아래 n행 미리보기 (기본 5) |
| `df.shape` | (행 수, 열 수) |
| `df.columns` | 열 이름 목록 |
| `df.dtypes` | 열별 자료형 |
| `df.info()` | 열·자료형·결측 요약 한 방에 |
| `df.describe()` | 숫자 열의 통계 요약(개수·평균·최소·최대 등) |

In [ ]:
# [제공 코드] 오늘 내내 쓸 라이브러리를 불러옵니다.
# 판다스는 관례상 pd, 넘파이(numpy)는 np 라는 별명으로 씁니다.
import pandas as pd
import numpy as np

In [ ]:
# 방법 1) 딕셔너리로 작은 데이터프레임을 직접 만들어 봅니다.
# 키(key)가 열 이름, 값(리스트)이 그 열의 데이터가 됩니다.
menu_df = pd.DataFrame({
    "음료": ["아메리카노", "카페라떼", "레모네이드"],
    "가격": [3500, 4500, 5000],
    "카테고리": ["커피", "커피", "에이드"],
})
menu_df

In [ ]:
# 방법 2) 실무의 기본 — CSV 파일을 통째로 읽어 데이터프레임으로.
# (한글이 깨지면 encoding 을 바꿔 보세요: 윈도우 엑셀이 만든 CSV 는 encoding="cp949" 가 많습니다.)
df = pd.read_csv("data/cafe_sales.csv")
df.head()   # 위에서 5행만 미리보기

In [ ]:
# 표의 크기와 구조 파악하기
print("행·열 크기:", df.shape)      # (60, 6) — 60행 6열
print("열 이름:", list(df.columns))
print("--- 열별 자료형 ---")
print(df.dtypes)

In [ ]:
# info() — 열 이름·자료형·결측 개수를 한 번에.
# 아래 출력을 보면 '수량' 열만 55 non-null (60행 중 5건이 비어 있음),
# '가격' 열은 숫자 같지만 자료형이 문자열(str)입니다. 왜일까요? (오후에 정제합니다)
# 참고: 판다스는 문자열 열을 'object' 로 표시합니다 (info()/dtypes 에 보이는 object = 문자열).
df.info()

In [ ]:
# describe() — 기본은 '숫자 열'의 통계 요약(개수·평균·표준편차·최소·사분위·최대).
# '수량'의 평균이 39.09?! 최댓값이 999라서 평균이 크게 부풀었어요.
# (999는 잘못 입력된 이상치 — 오후에 처리합니다. 중앙값 50%는 3으로 멀쩡하죠.)
display(df.describe())

In [ ]:
# 범주형(문자열) 열의 요약은 숫자 열을 빼고 봅니다 — describe(exclude='number').
# count(개수) · unique(고유값 수) · top(최빈값) · freq(최빈값이 나온 횟수) 를 보여 줍니다.
# (숫자형과 항목이 다릅니다 — 범주형엔 평균·사분위가 없으니까요.)
display(df.describe(exclude='number'))

# 수치형·범주형을 한 표에 함께 보려면 include='all' 을 줍니다(빈 칸은 NaN 으로 표시).
display(df.describe(include='all'))

### 🖐️ 함께 따라하기 — 데이터 첫인사 (동네 서점 데이터)
데모는 카페였죠? 이제 **데모와 다른 가게(동네 서점) 판매 데이터**를 직접 불러와, 같은 기술을 여러분 손으로 연습합니다. 아래 셀의 주석 순서대로 강사와 함께 타이핑해 보세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# ※ 데모는 '카페' 데이터였죠. 이번엔 데모와 다른 가게 데이터(동네 서점)로 같은 기술을 연습합니다.
# 1) pd.read_csv 로 data/bookstore_sales.csv 를 읽어 변수 books 에 담는다
# 2) books 의 크기(shape)를 출력한다
# 3) 앞 3행(head)과 뒤 3행(tail)을 각각 확인한다
# 4) info() 로 열·자료형·결측 요약을 본다

### ✅ 바로 확인 퀴즈
**1.** `df.shape` 가 `(60, 6)` 을 돌려줬습니다. 6은 무엇의 개수일까요?

<details>
<summary>정답 보기</summary>

**열(column)의 개수**입니다. `shape` 는 `(행 수, 열 수)` 순서예요. 즉 60행 6열.

</details>

**2.** 표의 위쪽 5행만 빠르게 보고 싶습니다. 어떤 메서드를 쓸까요?

<details>
<summary>정답 보기</summary>

`df.head()` 입니다. 괄호에 숫자를 넣으면(`df.head(3)`) 그 개수만큼 봅니다. 아래쪽은 `df.tail()`.

</details>

---
# 2. 열 선택 — 시리즈와 데이터프레임

## 왜 필요할까요?
표에서 **필요한 열만** 꺼내 쓰는 일은 분석의 첫걸음입니다. 대괄호 `[]` 로 고릅니다.

> **핵심 구분** — 대괄호를 **한 겹**으로 쓰면 열 하나가 **시리즈**로,
> **두 겹**(리스트)으로 쓰면 여러 열이 **데이터프레임**으로 나옵니다.

| 문법 | 결과 | 모양 |
|---|---|---|
| `df['음료']` | 시리즈 (1차원) | 세로 한 줄 |
| `df[['음료', '가격']]` | 데이터프레임 (2차원) | 표 |

열 하나여도 표로 받고 싶으면 `df[['음료']]` 처럼 **대괄호 두 겹**을 씁니다.

<img src="images/series_vs_dataframe.png" alt="시리즈 vs 데이터프레임" width="720">

In [ ]:
# 대괄호 한 겹 -> 시리즈(Series)
drinks = df['음료']
print(type(drinks))       # <class 'pandas.Series'>
print(drinks.head(3))

In [ ]:
# 대괄호 두 겹(리스트) -> 데이터프레임(DataFrame)
two_cols = df[['음료', '가격']]
print(type(two_cols))     # <class 'pandas.DataFrame'>
two_cols.head(3)

### 🖐️ 함께 따라하기 — 열 골라내기
분야 열 하나, 그리고 도서명·정가·판매부수 세 열을 각각 뽑아 봅시다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) '분야' 열 하나를 시리즈로 뽑아 앞 5개를 출력한다
# 2) 그 결과의 type 을 출력해 Series 임을 확인한다
# 3) '도서명','정가','판매부수' 세 열을 데이터프레임으로 뽑아 head() 를 본다
# 4) 그 결과의 type 을 출력해 DataFrame 임을 확인한다

### 열·행 버리기와 이름 바꾸기 — `drop` · `rename`

필요 없는 **열**이나 특정 **행**은 `drop` 으로 버립니다. 열은 `columns=`, 행은 `index=` 로 지정합니다. 지저분한 **열 이름**은 `rename(columns={{옛이름: 새이름}})` 으로 정리합니다.

| 코드 | 뜻 |
|---|---|
| `df.drop(columns=['회원여부'])` | 회원여부 **열**을 버린 새 표 |
| `df.drop(index=[0, 1])` | 0·1번 **행**을 버린 새 표 |
| `df.rename(columns={{'음료': '메뉴'}})` | 열 이름 음료 → 메뉴 |

In [ ]:
# 필요 없는 열/행 버리기, 열 이름 바꾸기 (원본은 그대로, 결과만 확인)
cafe = pd.read_csv('data/cafe_sales.csv')
no_member = cafe.drop(columns=['회원여부'])
print('열 개수:', len(cafe.columns), '→', len(no_member.columns))
no_rows = cafe.drop(index=[0, 1])                # 0, 1번 행(2개)을 버린 새 표
print('행 개수:', len(cafe), '→', len(no_rows))
renamed = cafe.rename(columns={'음료': '메뉴', '주문일': '날짜'})
print('바뀐 열 이름:', list(renamed.columns))

### ✅ 바로 확인 퀴즈
**1.** `df['가격']` 와 `df[['가격']]` 의 결과 자료형은 각각 무엇인가요?

<details>
<summary>정답 보기</summary>

앞은 **시리즈(Series)**, 뒤(대괄호 두 겹)는 **데이터프레임(DataFrame)** 입니다.

</details>

**2.** 음료와 회원여부 두 열만 표로 보려면?

<details>
<summary>정답 보기</summary>

`df[['음료', '회원여부']]` — 열 이름을 **리스트**로 묶어 대괄호 두 겹으로 넣습니다.

</details>

---
# 3. 행 고르기 — loc 와 iloc

## 왜 필요할까요?
열이 아니라 **특정 행**(또는 행과 열의 교차점)을 골라야 할 때가 많습니다. 두 도구가 있어요.

> **한 줄 요약** — `loc` 는 <strong>이름표(라벨)</strong>로, `iloc` 는 <strong>순서(위치 번호)</strong>로 고릅니다.
> `i` 는 integer(정수 위치)의 i 라고 외우세요.

| 문법 | 의미 |
|---|---|
| `df.loc[0, '음료']` | 라벨 0번 행, '음료' 열의 값 |
| `df.iloc[0, 1]` | 0번째 행, 1번째 열의 값 (위치) |
| `df.loc[0:2, ['음료','가격']]` | 0~2 라벨 행 × 두 열 |
| `df.iloc[0:3, 0:2]` | 앞 3행 × 앞 2열 |

> ⚠️ **슬라이싱 차이** — `loc[0:2]` 는 **끝(2)을 포함**(3개), `iloc[0:3]` 은 **끝(3)을 제외**(3개). 파이썬 리스트 슬라이싱과 같은 규칙은 `iloc` 쪽이에요.

<img src="images/loc_vs_iloc.png" alt="loc 와 iloc 비교" width="720">

In [ ]:
# loc — 라벨로 한 칸 집어내기
print("0번 행의 음료:", df.loc[0, '음료'])
print("4번 행의 카테고리:", df.loc[4, '카테고리'])

In [ ]:
# iloc — 위치(정수)로 집어내기
print("0번째 행, 1번째 열:", df.iloc[0, 1])   # 첫 행의 음료
print("--- 첫 3행 × 앞 2열 ---")
print(df.iloc[0:3, 0:2])

In [ ]:
# loc 로 여러 행 × 여러 열 교차 선택 (끝 라벨 2 포함 -> 3행)
df.loc[0:2, ['음료', '가격', '수량']]

### 🖐️ 함께 따라하기 — 행과 칸 집어내기
라벨과 위치, 두 방식으로 같은 데이터를 꺼내 보며 차이를 몸에 익힙시다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) loc 로 5번 행의 '도서명' 값을 출력한다
# 2) iloc 로 맨 마지막 행 전체를 출력한다 (힌트: 위치 -1)
# 3) loc 로 0~4번 행의 '도서명','분야','적립회원' 세 열을 표로 본다
# 4) iloc 로 앞 5행 × 앞 3열을 표로 본다

### ✅ 바로 확인 퀴즈
**1.** `df.iloc[0:3]` 은 몇 개의 행을 돌려줄까요?

<details>
<summary>정답 보기</summary>

**3개**(0·1·2번). `iloc` 슬라이싱은 파이썬 리스트처럼 **끝을 제외**합니다.

</details>

**2.** 위치가 아니라 **열 이름**으로 행·열을 고르려면 `loc` 와 `iloc` 중 무엇을 쓸까요?

<details>
<summary>정답 보기</summary>

**`loc`** 입니다. `loc` 는 라벨(이름표) 기반, `iloc` 는 정수 위치 기반이에요.

</details>

---
# 4. 조건으로 걸러내기 — 불리언 필터

## 왜 필요할까요?
"가격이 5000원 이상인 주문만", "회원이면서 커피인 주문만" 처럼 **조건에 맞는 행**만 뽑는 건 분석의 핵심입니다.

> **원리** — 조건식(`df['가격'] >= 5000`)은 각 행마다 참/거짓(True/False) 시리즈를 만듭니다.
> 그 참/거짓 시리즈를 다시 `df[...]` 에 넣으면 **True 인 행만** 남습니다.

| 문법 | 의미 |
|---|---|
| `df[df['카테고리'] == '커피']` | 커피인 행만 |
| `df[(A) & (B)]` | A **그리고** B (둘 다 참) |
| `df[(A) \| (B)]` | A **또는** B (하나라도 참) |
| `df[~(A)]` | A 가 **아닌** 행 (부정) |
| `df[df['음료'].isin([...])]` | 목록 중 하나면 |
| `df[df['수량'].between(2, 4)]` | 2 이상 4 이하 |

> ⚠️ **괄호 필수** — 조건 두 개를 `&`·`|` 로 묶을 때는 각 조건을 **반드시 소괄호** 로 감싸세요. `and`·`or` 가 아니라 `&`·`|`·`~` 를 씁니다.

<img src="images/boolean_filter.png" alt="불리언 필터 3단계" width="760">

In [ ]:
# 단일 조건 — 카테고리가 커피인 행만
coffee = df[df['카테고리'] == '커피']
print("커피 주문 건수:", coffee.shape[0])   # 40건
coffee.head(3)

In [ ]:
# 두 조건 결합 — & 는 '그리고', | 는 '또는', ~ 는 '아닌'
vip_coffee = df[(df['회원여부'] == 'Y') & (df['카테고리'] == '커피')]
print("회원이면서(&) 커피:", vip_coffee.shape[0], "건")   # 21건

# | 는 '또는' — 회원이거나(Y) 커피이거나, 둘 중 하나만 맞아도 (각 조건은 소괄호로!)
member_or_coffee = df[(df['회원여부'] == 'Y') | (df['카테고리'] == '커피')]
print("회원이거나(|) 커피:", member_or_coffee.shape[0], "건")

# ~ 는 '아닌' — 커피가 아닌(에이드·논커피) 행
not_coffee = df[~(df['카테고리'] == '커피')]
print("커피가 아닌(~) 주문:", not_coffee.shape[0], "건")   # 20건

In [ ]:
# isin — 목록에 속하면 / between — 범위 안이면
ades = df[df['카테고리'].isin(['에이드', '논커피'])]
print("에이드 또는 논커피:", ades.shape[0], "건")     # 20건

mid_qty = df[df['수량'].between(2, 4)]
print("수량 2~4개 주문:", mid_qty.shape[0], "건")      # 35건

### 🖐️ 함께 따라하기 — 원하는 책만 골라내기
여러 조건을 조합해 서점 데이터를 필터링해 봅시다. 괄호를 빠뜨리지 않도록 주의!

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 판매부수가 4 이상인 행만 골라 건수를 출력한다
#    (정가는 콤마가 섞인 문자열이라 아직 숫자 비교엔 못 써요 — 오후에 정제합니다)
# 2) 적립회원('Y')이면서 분야가 '소설'인 행만 골라 head() 를 본다
# 3) 분야가 '소설' 또는 '에세이'인 행만 isin 으로 골라 건수를 출력한다
# 4) 적립회원이 아닌(~) 행만 골라 건수를 출력한다

### ✅ 바로 확인 퀴즈
**1.** `df[df['가격'] >= 5000 & df['수량'] >= 2]` 가 에러를 냅니다. 왜일까요?

<details>
<summary>정답 보기</summary>

각 조건을 **소괄호**로 감싸지 않아서예요. `df[(df['가격'] >= 5000) & (df['수량'] >= 2)]` 로 고쳐야 합니다. `&` 는 비교보다 우선순위가 높아 괄호가 없으면 엉뚱하게 묶입니다.

</details>

**2.** "커피가 **아닌**" 행만 고르는 두 가지 방법을 써 보세요.

<details>
<summary>정답 보기</summary>

`df[df['카테고리'] != '커피']` 또는 `df[~(df['카테고리'] == '커피')]`. `~` 는 참/거짓을 뒤집는 부정 연산자예요.

</details>

---
# 5. 정렬 — sort_values

## 왜 필요할까요?
"가장 많이 팔린 순", "최근 주문 순" 처럼 **순서대로 줄 세우면** 데이터가 훨씬 잘 읽힙니다.

| 문법 | 의미 |
|---|---|
| `df.sort_values('수량')` | 수량 **오름차순**(작은→큰) |
| `df.sort_values('수량', ascending=False)` | **내림차순**(큰→작은) |
| `df.sort_values(['카테고리', '수량'], ascending=[True, False])` | 여러 기준(카테고리 오름 → 같으면 수량 내림) |
| `df.sort_index()` | **인덱스(행 번호)** 기준 정렬 |

> 정렬은 **새 표를 돌려줄 뿐** 원본을 바꾸지 않습니다. 결과를 쓰려면 변수에 다시 담으세요. 결측치(NaN)는 기본적으로 **맨 뒤**로 갑니다.

In [ ]:
# 단일 열 정렬 — 수량이 많은 순(내림차순) 상위 5
# 맨 위 두 건의 수량이 999?! 잘못 입력된 이상치입니다 (오후에 처리해요).
df.sort_values('수량', ascending=False).head(5)[['음료', '카테고리', '수량']]

In [ ]:
# 여러 기준 정렬 — 카테고리 오름차순, 같은 카테고리 안에서는 수량 내림차순
df.sort_values(['카테고리', '수량'], ascending=[True, False]).head(5)[['카테고리', '음료', '수량']]

In [ ]:
# 주문일로 정렬 (문자열이지만 'YYYY-MM-DD' 형식이라 날짜순과 같음)
# sort_index() 로 다시 원래 행 번호 순서로 되돌릴 수도 있습니다.
by_date = df.sort_values('주문일')
print(by_date.head(3)[['주문일', '음료']])
print("--- 인덱스 기준 원위치 ---")
print(by_date.sort_index().head(3)[['주문일', '음료']])

### 🖐️ 함께 따라하기 — 줄 세우기
서점 데이터를 여러 기준으로 정렬해 보며 `ascending` 옵션에 익숙해집시다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) '판매부수' 오름차순으로 정렬해 앞 5행의 도서명·판매부수를 본다
# 2) '판매부수' 내림차순으로 정렬해 앞 5행을 본다
# 3) '적립회원' 오름차순 -> 같으면 '판매부수' 내림차순, 두 기준으로 정렬해 head() 를 본다
# 4) 결과를 새 변수에 담아 원본 books 는 그대로임을 head() 로 확인한다

### 정렬·필터 뒤 인덱스 정리 — `reset_index` · `set_index`

정렬하거나 조건으로 걸러내면 **행 내용은 그대로지만 인덱스(행 번호)가 뒤죽박죽**이 됩니다. 0, 1, 2… 로 새로 매기려면 `reset_index(drop=True)` 를 씁니다(`drop=True` 는 옛 인덱스를 새 열로 남기지 않겠다는 뜻이에요). 반대로 특정 열을 **행의 이름표**(라벨)로 삼고 싶으면 `set_index('열')` 을 씁니다.

In [ ]:
# 정렬하면 인덱스가 뒤섞인다 -> reset_index 로 0,1,2... 로 다시 매기기
top = df.sort_values('수량', ascending=False).head(5)
print("정렬 직후 인덱스:", list(top.index))         # 원래 행 번호라 들쭉날쭉
tidy = top.reset_index(drop=True)
print("reset_index 후 인덱스:", list(tidy.index))    # 0,1,2,3,4

# set_index — 특정 열을 행 라벨(이름표)로 삼기
by_drink = df.head(3).set_index('음료')
print("--- '음료'를 인덱스로 삼은 표 ---")
print(by_drink[['카테고리', '수량']])

### 중복 제거 — `drop_duplicates`

같은 값이 여러 번 나오는 열에서 **중복을 빼고 한 번씩만** 보고 싶을 때 `drop_duplicates` 를 씁니다. 특정 열 기준으로 첫 등장만 남기려면 `subset` 을 줍니다.

In [ ]:
# 음료 종류를 중복 없이 (첫 등장만 남기기)
cafe = pd.read_csv('data/cafe_sales.csv')
unique_drinks = cafe.drop_duplicates(subset='음료')
print('전체', len(cafe), '행 → 음료 기준 중복 제거', len(unique_drinks), '행')

### ✅ 바로 확인 퀴즈
**1.** 수량이 **많은 순**으로 정렬하려면 어떤 옵션을 줄까요?

<details>
<summary>정답 보기</summary>

`df.sort_values('수량', ascending=False)`. `ascending=False` 가 내림차순(큰 값이 위)입니다.

</details>

**2.** `df.sort_values('수량')` 을 실행한 뒤 `df` 를 다시 보면 순서가 바뀌어 있을까요?

<details>
<summary>정답 보기</summary>

아니요. `sort_values` 는 **정렬된 새 표를 돌려줄 뿐** 원본을 바꾸지 않습니다. 바꾸려면 `df = df.sort_values(...)` 처럼 다시 담아야 해요.

</details>

---
## 🚀 응용 클론코딩 — 가장 많이 팔린 책 찾기 → 이상치 발견

오늘 배운 것을 **한 흐름**으로 이어 봅시다: 열 선택 → 정렬 → 조건 필터.

**미션**: 서점 데이터에서 "**도서명·분야·판매부수**" 세 열만 남기고, "**판매부수가 많은 순**"으로 줄 세워 상위 5권을 봅니다.

판매부수 내림차순으로 정렬하면 맨 위에 판매부수 999짜리 책이 올라옵니다. "어? 책 한 권이 999부?" — 이렇게 **정렬만 해도 이상치(잘못 입력된 값)가 눈에 띕니다.** 그래서 이 흐름의 진짜 목적은 베스트셀러 순위 자체가 아니라, **줄 세워 이상한 데이터를 발견**하는 것입니다. 999를 빼고 다시 보면 진짜 많이 팔린 책이 보이죠. 이 정제 작업이 바로 다음 절이에요.

In [ ]:
# 🖐️ 함께 따라하기 — 가장 많이 팔린 책 찾기 (아래 순서대로 직접 작성해 보세요)
# 1) '도서명','분야','판매부수' 세 열만 남긴 표를 top 에 담는다
# 2) '판매부수' 내림차순으로 정렬해 상위 5개를 본다  -> 맨 위 숫자가 이상하지 않나요?
# 3) 판매부수가 999 가 아닌 행만 남겨(불리언 필터) 다시 상위 5개를 본다
#    (999 는 정상 판매가 아니라 입력 오류 = 이상치! 다음 절에서 제대로 다룹니다)

---
## 이번 강의 정리

| 하고 싶은 일 | 도구 |
|---|---|
| 표 훑어보기 | `head`·`tail`·`shape`·`info`·`describe` |
| 열 하나 뽑기 (시리즈) | `df['열']` |
| 여러 열 뽑기 (표) | `df[['열1','열2']]` |
| 라벨로 행·칸 고르기 | `df.loc[행, 열]` |
| 위치로 행·칸 고르기 | `df.iloc[행, 열]` |
| 조건으로 행 거르기 | `df[df['열'] 조건]`, `&`·`\|`·`~`, `isin`, `between` |
| 정렬하기 | `df.sort_values(...)`, `sort_index()` |

## ⏭️ 예고 — 다음 시간: 지저분한 데이터 정제하기
오늘 훑어보다 보니 이상한 점이 많았죠?
- **수량 999** 짜리 이상치, **비어 있는(NaN)** 수량 5건
- **가격이 문자열** (`"5,000"` 처럼 콤마가 섞여 숫자로 계산이 안 됨)
- **음료 이름에 앞뒤 공백** (`"  레모네이드 "`)

다음 시간에는 이 **결측치·이상치·자료형·문자열**을 말끔히 정제하고, **총액·요일** 같은 새 정보(파생 변수)를 만들어 봅니다. 그렇게 깨끗해진 데이터로 다음 단원에서는 **EDA·시각화**(그룹별 집계·표 병합·matplotlib 그래프)로 나아갑니다.